# Show-o2: text-derived red steering

**Can a text-derived red steering signal increase red elements in generated images?**

Select a GPU runtime (A100 / High RAM when available). Run setup once, restart
the session, then load the model. Push this branch before using it in Colab.

Implemented: baseline generation and text-only direction extraction.
Image-position injection is the next step. Real Show-o2 GPU execution remains
unverified. The general demo stays in `colab.ipynb`.


In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)


## 1. Setup

Run once in a fresh Colab runtime. Use our experiment branch and a fixed upstream commit.


In [ ]:
import subprocess

branch = "experiment/text-red-steering"
commit = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"

subprocess.run(["git", "clone", "--branch", branch,
                "https://github.com/Nicerova7/show-o2-space.git",
                "/content/show-o2-space"], check=True)
subprocess.run(["git", "clone", "https://github.com/showlab/Show-o.git",
                "/content/Show-o"], check=True)
subprocess.run(["git", "-C", "/content/Show-o", "checkout", commit], check=True)


In [ ]:
# Keep Colab's matched PyTorch and torchvision installation.
get_ipython().run_line_magic("pip", 'install transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1')


### Load the model

Select **Runtime > Restart session**, then continue here. Run model loading once;
reuse `adapter` for the following experiments.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter

adapter = ShowO2Adapter()
print(torch.cuda.get_device_name(0), adapter.dtype)


## 2. Generation defaults and baseline

Edit `default_kwargs` to control Show-o2 generation. `model` is our loaded adapter;
`steps` controls flow sampling, `guidance` controls CFG, and `seed` fixes the initial noise.
Run this cell again when changing defaults, then regenerate the baseline.


In [ ]:
model = adapter
default_kwargs = dict(model=model, steps=50, guidance=5.0, seed=42)

def generate_image(prompt, model, steps, guidance, seed):
    return model.generate(prompt, steps=steps, guidance=guidance, seed=seed)

prompt = "A photograph of a car parked on a street."
output = PROJECT / "outputs/steering-red"
output.mkdir(parents=True, exist_ok=True)


In [ ]:
baseline = generate_image(prompt, **default_kwargs)
baseline.save(output / "baseline.png")
metadata = dict(
    model=MODEL_ID, prompt=prompt,
    **{key: default_kwargs[key] for key in ("steps", "guidance", "seed")},
    dtype=str(model.dtype), stage="baseline_only",
    upstream_revision=subprocess.check_output(
        ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
    ).strip(),
    project_revision=subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip(),
)
(output / "baseline.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
display(baseline)


## 3. ActAdd: red minus whitespace

Use the Show-o2 backbone with `positive_text="red"`, `negative_text=" "`, and `c=5`.
Prepend the same BOS, then right-pad the shorter token sequence with space tokens,
as in ActAdd. The cell prints token IDs and added padding; equal lengths add nothing.
Space tokens remain visible to causal attention, rather than being masked as PAD.

Capture the residual stream **entering** decoder block `layer` (zero-based), then
subtract every aligned position. No averaging, last-token selection, or normalization.
The result has shape `[1, aligned_sequence_length, hidden_dimension]`.


In [ ]:
from experiments.steering_utils import align_prompt_tokens, capture_layer_input

@torch.no_grad()
def extract_text_hidden(token_ids, model, layer):
    input_ids = torch.tensor([token_ids], device=model.device)
    length = input_ids.shape[1]
    causal_mask = torch.full(
        (length, length), float("-inf"), device=model.device, dtype=model.dtype
    ).triu(diagonal=1)[None, None]

    with capture_layer_input(model.layers, layer) as captured, model.autocast():
        model.model.showo.model(
            input_ids=input_ids, attention_mask=causal_mask,
            use_cache=False, output_hidden_states=False, return_dict=True,
        )
    return captured["hidden"]


In [ ]:
steering_kwargs = dict(
    layer=8,
    c=5.0,
    positive_text="red",
    negative_text=" ",
)
print("Available layers:", list(range(len(model.layers))))


In [ ]:
# Rerun after changing layer or either text.
layer = steering_kwargs["layer"]
red_ids, control_ids, padding = align_prompt_tokens(
    steering_kwargs["positive_text"], steering_kwargs["negative_text"],
    tokenizer=model.tokenizer, bos_id=model.token_ids["bos_id"],
)
print("red IDs / tokens:", red_ids, model.tokenizer.convert_ids_to_tokens(red_ids))
print("space IDs / tokens:", control_ids, model.tokenizer.convert_ids_to_tokens(control_ids))
print("Space tokens added on the right:", padding)

hidden_red = extract_text_hidden(red_ids, model=model, layer=layer)
hidden_control = extract_text_hidden(control_ids, model=model, layer=layer)
print("Aligned states:", hidden_red.shape, hidden_control.shape)

extraction_metadata = dict(
    layer=layer, positive_text=steering_kwargs["positive_text"],
    negative_text=steering_kwargs["negative_text"],
    red_token_ids=red_ids, control_token_ids=control_ids, padding=padding,
    capture="decoder block input (resid_pre)", alignment="tokens_right, space padding",
)


In [ ]:
# Rerun after changing c. Keep all aligned sequence positions.
c = steering_kwargs["c"]
h_red = hidden_red.float()
h_control = hidden_control.float()
v_red = h_red - h_control
steering_vector = c * v_red

print("h_red / h_control / v_red:", h_red.shape, h_control.shape, v_red.shape)
print("Difference norm at each token position:", v_red.norm(dim=-1))
print("c:", c, "Scaled vector norm:", steering_vector.norm().item())


`v_red` is the raw red-minus-whitespace activation difference, including the BOS
position. The common BOS should cancel under causal attention. `steering_vector`
is already scaled by `c`; do not multiply by `c` again during injection.
Changing `c` currently scales the tensor only; image injection is the next step.


In [ ]:
torch.save(
    dict(h_red=h_red, h_control=h_control, v_red=v_red,
         steering_vector=steering_vector,
         metadata=dict(**extraction_metadata, c=c, model=MODEL_ID)),
    output / "text_direction.pt",
)
print("Saved:", output / "text_direction.pt")


## 4. Inject during image generation (implementation pending)

The vector now contains all aligned source positions. For ActAdd-style injection,
choose a starting position `a` and add `steering_vector` to an equally long target
span at the **input** of the same decoder block. Never broadcast the entire
source sequence across every image position.

Mapping that span to Show-o2 image positions, choosing CFG rows, and selecting
flow steps remain explicit experiment decisions. The text alignment/extraction
above follows ActAdd; image generation is a multimodal adaptation.

First verify c=0 reproduces baseline. Then compare positive and negative c and
a matched-norm random-vector control. Always remove injection hooks after each run.


## 5. Compare and evaluate (implementation pending)

Display baseline and steered images side by side with identical prompts and seeds.
Report red pixel coverage as a simple proxy, alongside object preservation and
image quality: an overall red tint alone is not evidence of useful attribute control.
Choose layer/intensity on validation prompts, then evaluate held-out prompts and
multiple seeds. Save vectors, positions, layer, coefficients, sampling settings,
and model revision with the results. Pin checkpoint revisions before the study.
